# Validation / tests
Data-quality and pipeline-behaviour checks. Every check is `PASS` / `FAIL` / `INFO`. The notebook raises when any check fails (`fail_on_error = yes`), so the runner shows the run as failed.
Set `run_idempotency_test = yes` to also prove idempotency: it re-runs the latest PBS week and the WFP Silver MERGE and asserts that no table changes.

In [0]:
dbutils.widgets.dropdown("fail_on_error", "yes", ["yes", "no"])
dbutils.widgets.text("batch_id", "")
dbutils.widgets.dropdown("run_idempotency_test", "no", ["no", "yes"])

In [0]:
# ---- shared helpers (identical in every notebook) -------------------------------------------
import json, os, re, uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType, TimestampType)

CATALOG, SCHEMA = "workspace", "default"      # change here if your catalog / schema differ

def T(name):
    return f"{CATALOG}.{SCHEMA}.{name}"

AUDIT_TABLE = T("pipeline_execution_logs")
QUAR_TABLE = T("quarantine_records")

def esc(s):
    return str(s).replace("'", "''")

def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)

def norm_date(s):
    """Accepts 1.10.2026 / 01.10.2026 / 01-10-2026 / 2026-10-01 and returns dd.mm.yyyy (zero padded)."""
    s = str(s).strip()
    for fmt in ("%d.%m.%Y", "%d-%m-%Y", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(s, fmt).strftime("%d.%m.%Y")
        except ValueError:
            pass
    raise ValueError(f"Unrecognised date '{s}' (use dd.mm.yyyy)")

# ---- audit log ---------------------------------------------------------------------------------
AUDIT_SCHEMA = StructType([
    StructField("log_id", StringType()), StructField("batch_id", StringType()),
    StructField("load_type", StringType()), StructField("layer", StringType()),
    StructField("file_processed", StringType()), StructField("target_table", StringType()),
    StructField("start_time", TimestampType()), StructField("end_time", TimestampType()),
    StructField("status", StringType()),            # Success | Warning | Skipped | Failure
    StructField("rows_read", LongType()), StructField("rows_inserted", LongType()), StructField("rows_updated", LongType()),
    StructField("rows_quarantined", LongType()), StructField("error_message", StringType()),
])

def log_audit(batch_id, load_type, layer, file_processed, target_table, start_time, status,
              rows_read=0, rows_inserted=0, rows_updated=0, rows_quarantined=0, message=""):
    row = (str(uuid.uuid4()), batch_id, load_type, layer, str(file_processed), target_table,
           start_time, utcnow(), status, int(rows_read), int(rows_inserted), int(rows_updated), int(rows_quarantined),
           (message or "")[:1500])
    spark.createDataFrame([row], AUDIT_SCHEMA).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

# ---- quarantine --------------------------------------------------------------------------------
QUAR_ITEM_SCHEMA = StructType([StructField("record_ref", StringType()),
                               StructField("reason", StringType()),
                               StructField("record_json", StringType())])

def quarantine_df_from_items(items):
    """items = list of (record_ref, reason, record_json)"""
    return spark.createDataFrame(items, QUAR_ITEM_SCHEMA)

def write_quarantine(df, layer, source_key, batch_id):
    """Idempotent: clears the previous quarantine rows of the same layer+source, then appends.
    df needs the columns record_ref, reason, record_json."""
    spark.sql(f"DELETE FROM {QUAR_TABLE} WHERE layer = '{esc(layer)}' AND source_file = '{esc(source_key)}'")
    (df.select(F.expr("uuid()").alias("quarantine_id"), F.lit(batch_id).alias("batch_id"),
               F.lit(layer).alias("layer"), F.lit(source_key).alias("source_file"),
               "record_ref", "reason", "record_json", F.current_timestamp().alias("quarantined_at"))
       .write.format("delta").mode("append").saveAsTable(QUAR_TABLE))


In [0]:
results = []
def check(name, ok, detail=""):
    results.append((name, "PASS" if ok else "FAIL", str(detail)[:300]))

def info(name, detail):
    results.append((name, "INFO", str(detail)[:300]))

def cnt(sql):
    return spark.sql(sql).collect()[0][0]

TABLES = ["pipeline_execution_logs", "quarantine_records", "bronze_wfp_raw", "bronze_pbs_report", "bronze_pbs_annex",
          "silver_wfp_price_observation", "silver_quintile_index", "silver_city_price_observation"]

# 1) all tables exist
for t in TABLES:
    check(f"table exists: {t}", spark.catalog.tableExists(T(t)))
missing = [t for t in TABLES if not spark.catalog.tableExists(T(t))]
if missing:
    raise Exception(f"Tables missing, run Schemas first: {missing}")

# 2) audit log: no failures in the batch being validated
bid = dbutils.widgets.get("batch_id").strip()
if not bid:
    r = spark.sql(f"SELECT batch_id FROM {T('pipeline_execution_logs')} ORDER BY start_time DESC LIMIT 1").collect()
    bid = r[0][0] if r else ""
if bid:
    n_fail = cnt(f"SELECT count(*) FROM {T('pipeline_execution_logs')} WHERE batch_id = '{bid}' AND status = 'Failure'")
    check(f"audit: no Failure rows in batch {bid}", n_fail == 0, f"{n_fail} failures")
    for row in spark.sql(f"""SELECT status, count(*) c FROM {T('pipeline_execution_logs')}
                             WHERE batch_id = '{bid}' GROUP BY status""").collect():
        info(f"audit: status {row['status']} in batch", row["c"])

# 3) Bronze WFP
b = T("bronze_wfp_raw")
check("bronze_wfp_raw has rows", cnt(f"SELECT count(*) FROM {b}") > 0)
check("bronze_wfp_raw metadata never null",
      cnt(f"SELECT count(*) FROM {b} WHERE source_file IS NULL OR batch_id IS NULL OR load_timestamp IS NULL") == 0)
check("bronze_wfp_raw: one version per file (no duplicated loads)",
      cnt(f"SELECT count(*) FROM (SELECT source_file FROM {b} GROUP BY source_file HAVING count(DISTINCT source_file_signature) > 1)") == 0)

for t in ["bronze_wfp_raw", "bronze_pbs_report", "bronze_pbs_annex", "silver_wfp_price_observation",
          "silver_quintile_index", "silver_city_price_observation"]:
    check(f"load_timestamp never null: {t}", cnt(f"SELECT count(*) FROM {T(t)} WHERE load_timestamp IS NULL") == 0)

# 4) Silver WFP
s = T("silver_wfp_price_observation")
n_wfp = cnt(f"SELECT count(*) FROM {s}")
info("silver_wfp_price_observation rows", n_wfp)
check("silver_wfp: no duplicate business keys",
      cnt(f"""SELECT count(*) FROM (SELECT 1 FROM {s}
              GROUP BY country, market, commodity_name, unit, price_type, price_date HAVING count(*) > 1)""") == 0)
check("silver_wfp: price > 0, date/currency/market not null",
      cnt(f"SELECT count(*) FROM {s} WHERE price IS NULL OR price <= 0 OR price_date IS NULL OR currency IS NULL OR market IS NULL") == 0)
if n_wfp == 0:
    info("silver_wfp is empty", "OK for the 500-row Afghanistan sample with countries=Pakistan; load the full WFP file or set wfp_countries")

# 5) Silver quintile index
q = T("silver_quintile_index")
check("silver_quintile_index has rows", cnt(f"SELECT count(*) FROM {q}") > 0, "empty -> the old parser bug")
check("silver_quintile_index: exactly 6 rows (Q1-Q5 + Combined) per week",
      cnt(f"SELECT count(*) FROM (SELECT week_ending_date FROM {q} GROUP BY week_ending_date HAVING count(*) <> 6)") == 0)
check("silver_quintile_index: no duplicate (week, quintile)",
      cnt(f"SELECT count(*) FROM (SELECT 1 FROM {q} GROUP BY week_ending_date, quintile HAVING count(*) > 1)") == 0)
check("silver_quintile_index: index > 0 and not null",
      cnt(f"SELECT count(*) FROM {q} WHERE index_value IS NULL OR index_value <= 0") == 0)
check("silver_quintile_index: reported WoW % matches recomputed (tol 0.05)",
      cnt(f"SELECT count(*) FROM {q} WHERE abs((index_value / index_prev_week - 1) * 100 - pct_change_wow) > 0.05") == 0)
check("silver_quintile_index: reported YoY % matches recomputed (tol 0.05)",
      cnt(f"SELECT count(*) FROM {q} WHERE abs((index_value / index_prev_year - 1) * 100 - pct_change_yoy) > 0.05") == 0)

# 6) Silver city prices
c = T("silver_city_price_observation")
check("silver_city_price_observation has rows", cnt(f"SELECT count(*) FROM {c}") > 0)
check("silver_city: no duplicate (week, item, city)",
      cnt(f"SELECT count(*) FROM (SELECT 1 FROM {c} GROUP BY week_ending_date, item_sl, city_code HAVING count(*) > 1)") == 0)
check("silver_city: min <= avg <= max and dispersion >= 0",
      cnt(f"""SELECT count(*) FROM {c} WHERE (min_price IS NOT NULL AND min_price > avg_price + 0.005)
              OR (max_price IS NOT NULL AND max_price < avg_price - 0.005) OR price_dispersion_pct < 0""") == 0)
check("silver_city: 'city' holds city names, not item descriptions (<= 40 distinct)",
      cnt(f"SELECT count(DISTINCT city) FROM {c}") <= 40, cnt(f"SELECT count(DISTINCT city) FROM {c}"))

# 7) Bronze -> Silver reconciliation (every Bronze week reached Silver)
bw = {r[0] for r in spark.sql(f"SELECT DISTINCT week_ending_date FROM {T('bronze_pbs_report')}").collect()}
sw = {r[0] for r in spark.sql(f"SELECT DISTINCT date_format(week_ending_date, 'dd.MM.yyyy') FROM {q}").collect()}
check("every Bronze PBS week exists in Silver", bw <= sw, f"missing in Silver: {sorted(bw - sw)}")

# 8) quarantine overview
for row in spark.sql(f"SELECT layer, count(*) c FROM {T('quarantine_records')} GROUP BY layer ORDER BY layer").collect():
    info(f"quarantine rows ({row['layer']})", row["c"])

# 9) optional idempotency proof
if dbutils.widgets.get("run_idempotency_test") == "yes":
    snap = lambda: (cnt(f"SELECT count(*) FROM {T('bronze_pbs_report')}"), cnt(f"SELECT count(*) FROM {T('bronze_pbs_annex')}"),
                    cnt(f"SELECT count(*) FROM {q}"), cnt(f"SELECT count(*) FROM {c}"), cnt(f"SELECT count(*) FROM {s}"),
                    cnt(f"SELECT count(*) FROM {s} WHERE is_revised_flag"))
    latest = spark.sql(f"SELECT date_format(max(week_ending_date), 'dd.MM.yyyy') FROM {q}").collect()[0][0]
    before = snap()
    dbutils.notebook.run("bronzepbs", 1200, {"week_ending_dates": latest, "load_type": "full_backfill", "batch_id": "idempotency_test"})
    dbutils.notebook.run("silverpbs", 1200, {"week_ending_dates": latest, "load_type": "full_backfill", "batch_id": "idempotency_test"})
    if n_wfp > 0:
        dbutils.notebook.run("silverwfp", 1800, {"load_type": "full_backfill", "batch_id": "idempotency_test"})
    after = snap()
    check(f"idempotency: re-running week {latest} + WFP merge changes nothing", before == after, f"before={before} after={after}")

out = spark.createDataFrame(results, "check STRING, result STRING, detail STRING")
display(out)
n_fail = sum(1 for r in results if r[1] == "FAIL")
print(f"{sum(1 for r in results if r[1]=='PASS')} passed, {n_fail} failed")
if n_fail and dbutils.widgets.get("fail_on_error") == "yes":
    raise AssertionError(f"{n_fail} validation check(s) failed: " + "; ".join(r[0] for r in results if r[1] == "FAIL"))
dbutils.notebook.exit(json.dumps({"status": "Failure" if n_fail else "Success", "failed": n_fail}))